# Clustering Lab: K-Means + PCA on Real Data (Wine Chemistry)

**Goal:** Use **unsupervised learning** (K-Means) to discover meaningful groups in real chemical measurements of wines.  
You'll also use **PCA** to (i) visualize high-dimensional data in 2D and (ii) check if dimensionality reduction helps clustering.

**Real-world framing:** A quality-control team wants to **segment wines by their chemical profile** to tailor production and tasting protocols — without relying on labels.

**What you'll do**
- Load the **Wine** dataset (13 chemical features).
- Scale features and explore data quickly.
- Use **PCA** for 2D visualization and to understand variance explained.
- Run **K-Means**; select **k** via elbow & silhouette.
- Evaluate cluster quality with internal metrics and (optionally) compare to known cultivars (external metric, for teaching only).
- Interpret clusters by inspecting centroids in the original units.

Author: Matteo Cercola

In [ ]:
%pip install numpy pandas matplotlib seaborn openpyxl
%pip install ipywidgets

# === Setup & Imports ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

# For optional external validation mapping (Hungarian algorithm)
from scipy.optimize import linear_sum_assignment

## 1. Load Real Data & Define the Problem

We use the classic **Wine** dataset (chemical analysis of wines from 3 cultivars).  
Although we *have* labels (cultivar IDs), we will **not** use them to train K-Means — they are only for **external validation** at the end.

**Business-style problem:** *Segment wines into groups based on their chemical profiles so that the quality-control team can design differentiated protocols.*

In [ ]:
# Load the dataset
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y_true = pd.Series(wine.target, name="cultivar")  # kept aside for optional external validation

print("Shape:", X.shape)
X.head()

In [ ]:
X.columns

## 2. Feature Scales & Summary

Clustering is distance-based. **Always check scales** and **standardize** features when units differ.

In [ ]:
display(X.describe().T)

## 3. Preprocessing: Standardize Features

We use `StandardScaler` so that each feature has zero mean and unit variance, preventing larger-scale features from dominating distances.

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
X_std[:3]

## 4. PCA for Visualization and Variance Explained

We project data to **2 principal components** for visualization and compute **explained variance ratio (EVR)** to see how much variance is captured.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_std)
evr = pca.explained_variance_ratio_
print("Explained variance ratio (2 PCs):", evr, " -> cumulative:", evr.sum())

# Scatter in PCA space (no labels yet, purely unsupervised view)
plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1], s=18)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine data projected to 2D PCA space")
plt.show()

## 5. Choosing the Number of Clusters (*k*)

We use two standard tools:
- **Elbow method**: plot K-Means **inertia** (within-cluster sum of squares) vs *k* and look for the elbow.

$$
    \text{Inertia}(k) = \sum_{i=1}^k \sum_{x \in \mathcal{C}_i} \lVert x - \mu_i \rVert_2^2, \qquad \mu_i \text{ centroid of } \mathcal{C_i}
$$

- **Silhouette**: measures how similar a data point is to its own cluster compared to other clusters. It ranges from -1 to 1, where a higher value indicates better-defined clusters. It mainly balances cohesion (how close points are in a cluster) and separation (how far clusters are from each other).

$$
    s(x) = \frac{b(x) - a(x)}{ \max \{ a(x), b(x)\} }, \qquad 
    \begin{aligned} 
        a(x)& : \text{ average distance from } x \text{ to all other points in the same cluster} \\
        b(x)& : \text{ average distance from } x \text{ to all points in the next best cluster}
    \end{aligned}
$$

In [ ]:
ks = range(2, 11)
inertias = []
sil_scores = []

for k in ks:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_std)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_std, km.labels_))

# Elbow: inertia vs k
plt.figure(figsize=(6,4))
plt.plot(list(ks), inertias, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia (Within-Cluster SS)")
plt.title("Elbow Method")
plt.xticks(list(ks))
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Silhouette vs k
plt.figure(figsize=(6,4))
plt.plot(list(ks), sil_scores, marker="o")
plt.xlabel("k")
plt.ylabel("Average Silhouette Score")
plt.title("Silhouette vs k")
plt.xticks(list(ks))
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

best_k = ks[int(np.argmax(sil_scores))]
print("Best k by silhouette in this range:", best_k)

In [ ]:
# Let's check the number of cultivars in the true labels
print("True number of cultivars (classes):", y_true.nunique())

## 6. Fit K-Means

We'll pick the best *k* from the silhouette scan above

In [ ]:
k = best_k
km = KMeans(n_clusters=k, n_init="auto", random_state=42)
clusters = km.fit_predict(X_std)

In [ ]:
sil_pca = silhouette_score(X_std, clusters)

print("Silhouette score for k={}: {:.4f}".format(k, sil_pca))

## 7. Visualize Clusters in PCA Space

We plot K-Means clusters in the 2D PCA projection. We also show the same scatter colored by **true cultivars** (for teaching only) to compare.

In [ ]:
# Fit PCA again (2D) for clean plotting (already fit above, but okay to keep separate)
pca_plot = PCA(n_components=2, random_state=42).fit(X_std)
X_pca_plot = pca_plot.transform(X_std)

plt.figure(figsize=(12,5))
# Plot clusters
plt.subplot(1, 2, 1)
plt.scatter(X_pca_plot[:,0], X_pca_plot[:,1], c=clusters, s=20)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-Means Clusters (k={k}) in PCA Space")

# Plot true labels (teaching only)
plt.subplot(1, 2, 2)
plt.scatter(X_pca_plot[:,0], X_pca_plot[:,1], c=y_true, s=20)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("True Cultivars in PCA Space (for comparison)")
plt.show()

## 8.Does PCA Help Clustering?

Try running K-Means on the first **m** principal components only, then re-evaluate metrics.  
Sometimes reducing noise / collinearity helps clustering stability.

In [ ]:
U,S, VT = np.linalg.svd(X_std, full_matrices=False)

# Compute explained variance ratio
EV = (S**2) / (X_std.shape[0] - 1)
EVR = EV / EV.sum()
CVR = np.cumsum(EVR)


# Step 4: Create the scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(EV) + 1), CVR, marker='o', label='Cumulative Variance Ratio')
plt.axhline(y=0.90, color='red', linestyle='--', label='90% Variance Threshold')
plt.xlabel('Principal Component')
plt.ylabel('Cumulative Variance Ratio')
plt.xticks(range(1, len(EV) + 1))
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
K = np.argmax(CVR >= 0.90) + 1
K # Number of principal components required to retain at least 95% of the variance in the dataset

In [ ]:
pca_m = PCA(n_components=K, random_state=42)
X_pca_m = pca_m.fit_transform(X_std)

In [ ]:
km_pca = KMeans(n_clusters=k, n_init="auto", random_state=42)
cl_pca = km_pca.fit_predict(X_pca_m)

sil_pca = silhouette_score(X_pca_m, cl_pca)

print(f"K-Means on first {K} PCs -> Silhouette: {sil_pca:.3f}")

## 9. Takeaways

- **K-Means** can reveal structure in real chemical data without labels.
- **Scaling** is essential for distance-based clustering.
- **PCA** is useful to **visualize** clusters and sometimes to **improve** clustering by denoising / decorrelating features.
- Use **internal metrics** (Silhouette, CH, DB) to compare settings; use **external labels** only for teaching/validation when available.
- Centroid inspection in **original units** helps translate clusters into **actionable insights** (e.g., which chemical properties define each segment).